# pDockQ2 Interface Quality

This notebook demonstrates `run_pdockq2`, which scores the interface of a cofolded protein complex using the Zhu-2023 pDockQ2 formula. It combines per-residue pLDDT and the PAE matrix into a single scalar in `[0, 1]`. Germinal's final-filter stage rejects trajectories with `pdockq2 <= 0.23`.

Reference: Zhu et al., *Bioinformatics* 39:btad424 (2023). [doi:10.1093/bioinformatics/btad424](https://doi.org/10.1093/bioinformatics/btad424).

## Imports

The tool input requires a `Structure` whose B-factor column carries pLDDT (`b_factor_type=BFactorType.PLDDT` or `NORMALIZED_PLDDT`) and whose `metrics['pae']` contains the square PAE matrix emitted by the structure predictor.

In [ ]:
from proto_tools import PDockQ2Config, PDockQ2Input, PDockQ2Output, run_pdockq2
from proto_tools.tools.structure_scoring.pdockq2.pdockq2 import example_input

## Input API Reference

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structure` | `Structure` | *Required* | Cofolded complex with pLDDT in B-factors and PAE at `metrics['pae']`. |
| `binder_chain` | `str` | *Required* | Single-character chain ID of the binder. |
| `target_chains` | `list[str]` | *Required* | Target chain ID(s). Comma-separated strings are accepted and normalized. |

## Config API Reference

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `distance_cutoff` | `float` | `10.0` | CA-CA distance cutoff (Å) for interface residue detection. |

## Output API Reference

| Field | Type | Description |
|-------|------|-------------|
| `metrics.pdockq2` | `float` | Overall pDockQ2 score in `[0, 1]`. |
| `metrics.avg_interface_plddt` | `float` | Mean interface pLDDT (0-100) over scoring target chains. |
| `metrics.avg_interface_pae` | `float` | Mean normalized PAE over contact pairs. |
| `metrics.num_interface_contacts` | `int` | Cross-chain residue pairs within the cutoff. |
| `metrics.interfaces` | `list[InterfacePDockQ2]` | Per-chain breakdown for debugging. |

## Basic Usage

`example_input()` returns a bundled 2-chain fixture with hand-crafted PAE so the notebook runs offline. In real usage, the `Structure` and `metrics['pae']` come from AF3 / Chai-1 / Boltz-2 / Protenix output.

In [ ]:
inputs = example_input()
config = PDockQ2Config()
result: PDockQ2Output = run_pdockq2(inputs, config)

print(f"pdockq2:               {result.metrics.pdockq2:.4f}")
print(f"avg_interface_plddt:   {result.metrics.avg_interface_plddt:.2f}")
print(f"avg_interface_pae:     {result.metrics.avg_interface_pae:.4f}")
print(f"num_interface_contacts: {result.metrics.num_interface_contacts}")

## Per-Chain Interface Breakdown

The `interfaces` field exposes every chain's contribution, including those not aggregated into the final score. Useful when debugging multi-chain targets.

In [ ]:
for row in result.metrics.interfaces:
    print(
        f"chain {row.chain_id} | neighbors={row.neighbor_chains!r:5} | "
        f"if_plddt={row.if_plddt:6.2f} | norm_pae={row.norm_pae:.4f} | "
        f"pmidockq={row.pmidockq:.4f}"
    )

## Tightening the Distance Cutoff

The distance cutoff controls which CA-CA pairs count as "in contact". Tightening it drops marginal pairs and, if no contacts remain, the score falls to zero.

In [ ]:
tight = run_pdockq2(inputs, PDockQ2Config(distance_cutoff=4.0))
print(f"tight cutoff: pdockq2={tight.metrics.pdockq2:.4f}, contacts={tight.metrics.num_interface_contacts}")

## Export

Results can be exported as JSON for downstream analysis.

In [ ]:
from pathlib import Path

out_dir = Path("./example_output")
out_dir.mkdir(exist_ok=True)
result.export("pdockq2_result", export_path=out_dir, file_format="json")
print(f"Exported to {out_dir / 'pdockq2_result.json'}")